# Make my Satellite1 Ultra parts

This page builds a set of parts sized for **your** printer.

**You only need to do three things:**

1. Paste the code from the measuring step into the box below.
2. Click **Runtime → Run all** in the menu at the top.
3. Wait about 15 minutes, then your parts download automatically.

Nothing is installed on your computer. If a box asks you to allow something,
say yes — that is just Google letting this page run.


In [ ]:
#@title Step 1 — paste your code here, then press the ▶ button { display-mode: "form" }
MY_CODE = ""  #@param {type:"string"}

import base64, json
KEYS = ["xy_scale_correction_fraction","z_scale_correction_fraction","fastener_clearance_diameter_offset_mm","insert_bore_diameter_offset_mm","driver_cutout_diameter_offset_mm","passive_radiator_cutout_diameter_offset_mm","cable_passage_diameter_offset_mm","gasket_sheet_thickness_mm","gasket_compressed_thickness_offset_mm","active_driver_flange_thickness_mm","passive_radiator_flange_thickness_mm"]

if not MY_CODE.strip():
    raise SystemExit("Paste the code from the measuring step into the box above, then run this cell again.")

text = MY_CODE.strip()
if not text.startswith("S1U-"):
    raise SystemExit("That does not look like a code. It should start with S1U-")
try:
    padded = text[4:] + "=" * (-len(text[4:]) % 4)
    numbers = [float(v) for v in base64.b64decode(padded).decode().split(",")]
except Exception:
    raise SystemExit("That code looks damaged. Copy it again from the measuring step.")
if len(numbers) != len(KEYS):
    raise SystemExit("That code is incomplete. Copy it again from the measuring step.")

CALIBRATION = dict(zip(KEYS, numbers))
print("Your measurements were read correctly:\n")
for key, value in CALIBRATION.items():
    print(f"  {key:45s} {value}")


## Step 2 — everything below runs on its own

You do not need to change anything here. Just let it finish.

In [ ]:
#@title Install the CAD engine (about 3 minutes)
!pip install --quiet cadquery==2.6.1 pyyaml==6.0.2 numpy==2.2.6 2>&1 | tail -2
print("CAD engine ready.")


In [ ]:
#@title Fetch the Satellite1 Ultra design
!rm -rf /content/Satellite1-Ultra
!git clone --depth 1 --quiet https://github.com/BigPappy098/Satellite1-Ultra.git /content/Satellite1-Ultra
%cd /content/Satellite1-Ultra
!pip install --quiet --no-deps -e . 2>&1 | tail -1
print("Design files ready.")


In [ ]:
#@title Check your numbers are safe
import yaml
from satellite1_ultra.configuration import validate_physical_calibration

validate_physical_calibration(CALIBRATION)   # refuses anything physically implausible

with open("config/physical_calibration.yaml", "w") as handle:
    handle.write("# Generated from your measurements.\n")
    for key, value in CALIBRATION.items():
        handle.write(f"{key}: {value}\n")
print("Your numbers passed every safety check.")


In [ ]:
#@title Build your parts (about 10 minutes — this is the slow one)
import time
start = time.time()
from pathlib import Path
from satellite1_ultra.configuration import load_design_parameters
from satellite1_ultra.exporting import export_parts

parameters = load_design_parameters()
written = export_parts(Path("exports"), parameters)
print(f"\nBuilt {len(written)} files in {(time.time()-start)/60:.1f} minutes.")


In [ ]:
#@title Package your parts and download them
import shutil, os
from pathlib import Path
from satellite1_ultra.builder_files import (
    CALIBRATION_PRINT_ORDER, ULTRA_PRINT_ORDER, OFFICIAL_TOP_PRINT_ORDER)
from satellite1_ultra.official import OFFICIAL_PRINT_PARTS_REQUIRED

out = Path("/content/MY_SATELLITE1_ULTRA_PARTS")
shutil.rmtree(out, ignore_errors=True)
for folder, order in (("1_PRINT_THESE_TEST_PIECES_AGAIN", CALIBRATION_PRINT_ORDER),
                      ("2_ENCLOSURE_PARTS", ULTRA_PRINT_ORDER)):
    (out / folder).mkdir(parents=True, exist_ok=True)
    for source, friendly, _quantity in order:
        shutil.copy2(Path("exports/3mf") / f"{source}.3mf", out / folder / friendly)

# The six Satellite top parts are the official files, copied unchanged.
official = {part.name: part for part in OFFICIAL_PRINT_PARTS_REQUIRED}
(out / "3_SATELLITE_TOP_PARTS").mkdir(parents=True, exist_ok=True)
for source, friendly, _quantity in OFFICIAL_TOP_PRINT_ORDER:
    shutil.copy2(official[source].stl_path, out / "3_SATELLITE_TOP_PARTS" / friendly)

(out / "READ_ME.txt").write_text(
    "These parts are sized for your printer, from the measurements you entered.\n\n"
    "1_PRINT_THESE_TEST_PIECES_AGAIN - reprint and re-check any test piece you corrected.\n"
    "2_ENCLOSURE_PARTS               - the enclosure. Print every file.\n"
    "3_SATELLITE_TOP_PARTS           - the original Satellite1 top. Print all six.\n\n"
    "Then go back to the website and follow step 4 to build it.\n")

archive = shutil.make_archive("/content/MY_SATELLITE1_ULTRA_PARTS", "zip", out.parent, out.name)
print(f"Ready: {os.path.getsize(archive)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
    print("\nYour download should start now.")
except Exception:
    print("\nOpen the folder icon on the left and download MY_SATELLITE1_ULTRA_PARTS.zip")


## Done

Your parts are in **MY_SATELLITE1_ULTRA_PARTS.zip**.

If the download did not start, click the folder icon on the left-hand side and
download it from there.

Now go back to the website and follow **step 4 — Build it**.

*These files are generated from your measurements. Nothing has been physically
built and tested yet, so keep checking as you go.*
